# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data
df = pd.read_csv('/content/hotels.csv')

# 2. Handle missing values and drop target-leaking columns
cols_to_drop = ['reservation_status', 'reservation_status_date']
X = df.drop(columns=['is_canceled'] + cols_to_drop, errors='ignore')
y = df['is_canceled']

# 3. Train-test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. One-hot encode categorical features
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)

# Align columns
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 5. Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded.fillna(0))
X_test_scaled = scaler.transform(X_test_encoded.fillna(0))

print(f"Data prepared. X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")

Data prepared. X_train shape: (6582, 138), X_test shape: (2822, 138)


### ✍️ Your Response:
1. **How many total rows and columns are in the dataset?**
   The origina; dataset contains 9,404 rows and 30 columns in total. After preprocessing, the feature space is expanded to 130 columns and there are 2822 rows represending the 30% of the 70/30 split.

2. **What types of features (categorical, numerical) are included?**
The numerical include things like lead time, arrival date, children, parking spaces and revious cancellations. The categorical values include things like country, room type, and agent they booked through.

3. **What steps did you take to clean or prepare the data?**
   - Dropped columns that might leak important information that is private such as reservation_status and reservation_status_date. I also Split the data into a 70% training set and a 30% testing set. I also imputed remaining missing values with zero during alignment.
   Finally, I standardized the features using StandardScaler fitted exclusively on the training set to prevent data leakage.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Train Naïve Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred_nb = nb_model.predict(X_test_scaled)

print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Naïve Bayes Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.32      0.49      2111
           1       0.33      1.00      0.50       711

    accuracy                           0.49      2822
   macro avg       0.66      0.66      0.49      2822
weighted avg       0.83      0.49      0.49      2822

Confusion Matrix:
[[ 677 1434]
 [   3  708]]


### Reflection:
1. **How well does the model perform? And what metric is best used to judge the performance?**


2. **Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?**


### ✍️ Your Response:
1. **Performance:** Naïve Bayes provides a fast baseline, but has a high false positive rate (1,352 cases), indicating it over-predicts cancellations.
2. **Use Case:** It is best used for automated, low-risk real-time alerts or triggers such as sending automated payment verification links.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [8]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Train SVM with linear kernel
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred_svm = svm_model.predict(X_test_scaled)

print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2111
           1       0.81      0.82      0.81       711

    accuracy                           0.91      2822
   macro avg       0.88      0.88      0.88      2822
weighted avg       0.91      0.91      0.91      2822

Confusion Matrix:
[[1977  134]
 [ 130  581]]


### Reflection:
1. **How well does the model perform? And what metric is best used to judge the performance?**

2. **In what business situations could SVM provide better insights than simpler models?**

### ✍️ Your Response:
1. **Performance:** The SVM model achieved a high accuracy of **91%**, with a strong balance of precision (0.82) and recall (0.81) for cancellations.
2. **Use Case:** SVM is highly suited for strategic quarterly planning, financial risk forecasting, and overbooking strategy formulation. It is good for nonlinear relationships which is great for this model.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [11]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Create a validation split from the training data
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.15, random_state=42
)

# Train a Neural Network (MLPClassifier)
# Using 2 hidden layers (e.g., 64 and 32 neurons) with early stopping using validation data
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42
)

print("Training the Neural Network model... (this may take a moment)")
mlp_model.fit(X_train_final, y_train_final)

# Predict and evaluate on the test set
y_pred_mlp = mlp_model.predict(X_test_scaled)

print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_mlp))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

Training the Neural Network model... (this may take a moment)

Neural Network Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      2111
           1       0.85      0.81      0.83       711

    accuracy                           0.92      2822
   macro avg       0.89      0.88      0.89      2822
weighted avg       0.91      0.92      0.92      2822

Confusion Matrix:
[[2011  100]
 [ 137  574]]


### Reflection:
1. **How does this model compare to the others?**

2. **Would the business be comfortable using a “black box” model like this? Why or why not?**

### ✍️ Your Response:
1. **Comparison:** The Neural Network model is the top performer, achieving **92% accuracy** and the highest precision (**0.85**) for predicting cancellations.
2. **Business Comfort:** Yes, because the operational cost savings of a more accurate model outweigh the need for complete transparency, and modern feature attribution tools can mitigate the "black box" issue.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [10]:
from sklearn.metrics import accuracy_score

# Calculate accuracies
accuracy_nb = accuracy_score(y_test, y_pred_nb)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print("=== Model Accuracy Comparison ===")
print(f"Naïve Bayes Accuracy : {accuracy_nb * 100:.2f}%")
print(f"SVM Accuracy         : {accuracy_svm * 100:.2f}%")
print(f"Neural Network Acc   : {accuracy_mlp * 100:.2f}%")

=== Model Accuracy Comparison ===
Naïve Bayes Accuracy : 49.08%
SVM Accuracy         : 90.64%
Neural Network Acc   : 91.60%


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response:
1. **Model Evaluation Summary:**
  **Accuracy:** The Neural Network performed best with 91.60% accuracy, slightly beating SVM 90.64%
  **Training Time:** Naïve Bayes was practically instantaneous, while SVM and the Neural Network took longer to train, needing their own training sets.
  **Interpretability:** Naïve Bayes and SVM (linear coefficients) are much easier to interpret, while the Neural Network is a complex black-box model.
  **Ease of Use:** Naïve Bayes and SVM require less hyperparameter tuning than Neural Networks. For those reasons Neural Networks may be harder for stakeholders to understand and might not be the best choice.

2. **Deployment Recommendation:**
  I recommend deploying the SVM model. Although the Neural Network has a higher accuracy SVM offers nearly identical predictive power while being much easier to explain to stakeholders, lighter to maintain, and computationally less intensive to retraining on new datasets.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response:
1. **Business Recommendation:**
To conclude, I recommend implementing the SVM model for predicting cancellations. This model solves the hotel’s business problem of unpredictability in planning by accurately flagging bookings that are at high risk of cancellation. Operationally, this allows managers to safely overbook rooms with higher confidence and optimize staffing schedules. A minor risk is false positives, which can be mitigated by sending polite, automated pre-arrival verification reminders rather than canceling bookings abruptly.

2. **Learning Outcome Integration:**
This assignment connects to my learning outcome because I could use these models and the knowledge about which one is best for each scenario. I can use this when I work with business clients for myself to predict the likelyhood of their performance and future cash flows. Then, I'll have a better idea of how much they have disposable to invest.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"